In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd

from automed import AutoMed

[15:31:11] cuDF not found: falling back to standalone pandas.

In [2]:
titanic = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

In [3]:
autom = AutoMed()

print(autom.debug_load())
print(autom.json_pipeline())

None
{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropDateColumn', 'name': 'Drop date column', 'description': 'Step description...', 'configuration': {'ratio': {'description': "Description of the parameter's role", 'default': 0.8, 'value': 0.8}, 'random_state': {'description': "Description of the parameter's role", 'default': 12, 'value': 12}}, 'children': []}, {'step': 'ActOnehot', 'name': 'One hot encoding categorical features', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActTfIdf', 'name': 'TF-IDF', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActMeanColumn', 'name': 'Fill missing values with mean', 'description': 'Step description...', 'configuration': {'empty_threshold': {'description': 'Column with le

In [4]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [
        {
            'step': 'MetaStep',
            'tag': 'cleaning'
        },
        {
            'step': 'MetaStep',
            'tag': 'metric'
        },
        {
            'step': 'WrapKFold',
            'children': [{
                'step': 'ActSVMSVC'
            }]
        }
    ]
}

autom.load_pipeline(pipeline)
print(autom.json_pipeline())


{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropDateColumn', 'name': 'Drop date column', 'description': 'Step description...', 'configuration': {'ratio': {'description': "Description of the parameter's role", 'default': 0.8, 'value': 0.8}, 'random_state': {'description': "Description of the parameter's role", 'default': 12, 'value': 12}}, 'children': []}, {'step': 'ActOnehot', 'name': 'One hot encoding categorical features', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActTfIdf', 'name': 'TF-IDF', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActMeanColumn', 'name': 'Fill missing values with mean', 'description': 'Step description...', 'configuration': {'empty_threshold': {'description': 'Column with less or

In [5]:
print(titanic[['label']])
results = autom.fit(
    X=titanic.drop('label', axis=1),
    Y=titanic[['label']])
[ (r.pipeline.model, r.get_main_metric_value()) for r in results if r.pipeline.model is not None ]

     label
0        0
1        1
2        1
3        1
4        0
..     ...
886      0
887      1
888      0
889      1
890      0

[891 rows x 1 columns]


Output()

[15:31:12] running step: MetaOrderedStep (steps=WrapKFold,MetaStep)

           running step: MetaStep                                                                                  
           (steps=ActOnehot,ActDropNumericalColumn,ActDropDateColumn,ActDropTextualColumn,ActMeanColumn,ActSplitDat
           e,ActTfIdf)

[15:31:43] running step: MetaStep (steps=MetricSelection)

...

[15:31:47] running k-folds: ActSVMSVC (fold=2) (kernel=rbf, random_state=42, probability=False, class_weight=None)

[15:31:55] running k-folds: ActSVMSVC (fold=4) (kernel=rbf, random_state=42, probability=False, class_weight=None)

[15:31:59] running k-folds: ActSVMSVC (fold=5) (kernel=rbf, random_state=42, probability=False, class_weight=None)

[15:32:03] finishing cross-validation: ActSVMSVC (kernel=rbf, random_state=42, probability=False,                  
           class_weight=None)

[(<automed.actionables.learning.act_svm_svc.ActSVMSVC at 0x21de38b7750>,
  0.5359579167635433)]

In [8]:
print(results[0].pipeline.model)
print(results[0].pipeline.steps)

pm = results[0].pipeline.pickle()
[ len(r.pipeline.pickle()) for r in results ]

Learn : SVM Classification
[('Fill missing values with mean', <automed.actionables.cleaning.act_mean_column.ActMeanColumn object at 0x0000021DE38B7090>), ('One hot encoding categorical features', <automed.actionables.cleaning.act_onehot.ActOnehot object at 0x0000021DE38B70D0>), ('Transform string column to date', <automed.actionables.cleaning.act_split_date.ActSplitDate object at 0x0000021DE38B7210>), ('TF-IDF', <automed.actionables.cleaning.act_tf_idf.ActTfIdf object at 0x0000021DE38B7050>), ('Drop date column', <automed.actionables.cleaning.act_drop_date_column.ActDropDateColumn object at 0x0000021DE38B7010>), ('Drop Numerical Column', <automed.actionables.cleaning.act_drop_numerical_column.ActDropNumericalColumn object at 0x0000021DE38B7110>), ('Drop textual column', <automed.actionables.cleaning.act_drop_textual_column.ActDropTextualColumn object at 0x0000021DE38B7150>), ('Learn : SVM Classification', <automed.actionables.learning.act_svm_svc.ActSVMSVC object at 0x0000021DE38B7750>

[29605036]

In [9]:
import pickle

o = 200 # offset
n = 68  # # of samples
labels = titanic.iloc[o:(o+n)]['label']
predict_df = titanic.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)

results = m.predict(predict_df)
sum([ r == labels[o+i] for i, r in enumerate(results) ]) / n

0.6617647058823529

In [ ]:
final_boss_automed = AutoMed(max_workers=12)
final_boss_automed.debug_load()
final_boss_results = final_boss_automed.fit(titanic.drop('label', axis=1).copy(), titanic[['label']].copy())

[16:21:35] running step: MetaOrderedStep (steps=MetaExplorerStep,MetaStep)

           running step: MetaStep                                                                                  
           (steps=ActSplitDate,ActDropDateColumn,ActDropTextualColumn,ActTfIdf,ActOnehot,ActDropNumericalColumn,Act
           MeanColumn)

[16:22:39] running step: MetaStep (steps=ActRandomOverSampling,ActMinMaxScaler)

           running step: MetaStep (steps=MetricSelection)

           running step: MetaExplorerStep (steps=WrapGeneticGridSearch)

...

[16:55:56] running k-folds: ActXGBoost (fold=4)

[16:56:00] running k-folds: ActXGBoost (fold=3)

           running k-folds: ActXGBoost (fold=4)

[16:56:09] running k-folds: ActXGBoost (fold=4)

[16:56:32] finished all generations: KFold

In [ ]:
# sum([ len(r.model.pickle()) for r in final_boss_results ])
[ (r.model.ml_model, r.evaluate()) for r in final_boss_results if r.model.ml_model is not None ]

[(SVC(class_weight='balanced', kernel='linear', probability=True, random_state=42),
  {'accuracy': 0.8316301550436256,
   'balanced_accuracy': 0.8174481434159043,
   'classification_error': 0.18255185658409573,
   'f1_score': 0.7737527476536764,
   'precision': 0.7959759076688132,
   'recall': 0.7959759076688132,
   'specificity': 0.8779482902418684}),
 (RandomForestClassifier(max_depth=58, n_estimators=369, random_state=42),
  {'accuracy': 0.832741196409516,
   'balanced_accuracy': 0.8096114266862054,
   'classification_error': 0.19038857331379444,
   'f1_score': 0.7632567940672449,
   'precision': 0.8334999562305774,
   'recall': 0.8334999562305774,
   'specificity': 0.908907422852377}),
 (LogisticRegression(max_iter=3113, n_jobs=-1, random_state=42),
  {'accuracy': 0.8248948590797817,
   'balanced_accuracy': 0.8075738982542287,
   'classification_error': 0.19242610174577138,
   'f1_score': 0.7609766098257147,
   'precision': 0.7963635405492502,
   'recall': 0.7963635405492502,
   's